# Array Comparison Demo

Arrays require an **alignment** step before scoring: which gold element pairs with which
extracted element?

1. **Ordered (positional)** -- element 0 vs element 0. Default. Order matters.
2. **Key-field matching** -- match by a unique identifier. Order doesn't matter.

In [ ]:
from html import escape
from IPython.display import HTML, display
from struct_extract_eval import evaluate


def show_table(headers: list[str], rows: list[list]) -> None:
    status_colors = {
        "match": "#1a7f37", "mismatch": "#cf222e",
        "omission": "#9a6700", "hallucination": "#8250df",
    }
    th = "".join(
        f"<th style='text-align:left; padding:6px 12px; border-bottom:2px solid #d0d7de; background:#fff; color:#000;'>{escape(str(h))}</th>"
        for h in headers
    )
    trs = []
    for row in rows:
        tds = []
        for val in row:
            val_str = escape(str(val))
            color = status_colors.get(str(val), "#000")
            style = f"color:{color}; font-weight:bold;" if str(val) in status_colors else "color:#000;"
            tds.append(f"<td style='padding:4px 12px; border-bottom:1px solid #eaeef2; background:#fff; {style}'>{val_str}</td>")
        trs.append(f"<tr>{''.join(tds)}</tr>")
    display(HTML(f"""<table style="border-collapse:collapse; font-size:13px; font-family:monospace;">
        <thead><tr>{th}</tr></thead><tbody>{''.join(trs)}</tbody></table>"""))


def show_record(record) -> None:
    display(HTML(f"<b>Record {record.record_id}</b> &mdash; "
                 f"F1: {record.f1:.3f} &nbsp; P: {record.precision:.3f} &nbsp; R: {record.recall:.3f}"))
    show_table(
        ["Path", "Gold", "Extracted", "Score", "Status"],
        [
            [fr.path, str(fr.gold_value)[:25], str(fr.extracted_value)[:25],
             f"{fr.score:.1f}", fr.status]
            for fr in record.field_results
        ],
    )

## Data

Three records with process steps. The extracted arrays have the right elements but
in a **different order** from gold, plus realistic errors.

In [ ]:
GOLD = [
    {"steps": [
        {"name": "deposit", "temp": 600},
    ]},
    {"steps": [
        {"name": "deposit", "temp": 300},
        {"name": "anneal", "temp": 500},
    ]},
    {"steps": [
        {"name": "deposit", "temp": 300},
        {"name": "anneal", "temp": 500},
        {"name": "clean", "temp": 100},
    ]},
]

EXTRACTED = [
    {"steps": [
        {"name": "deposit", "temp": 600},
    ]},
    {"steps": [
        {"name": "anneal", "temp": 480},    # swapped + wrong temp
        {"name": "deposit", "temp": 300},   # swapped but correct
        {"name": "cool", "temp": 50},       # hallucinated
    ]},
    {"steps": [
        {"name": "anneal", "temp": 500},    # swapped but correct
        {"name": "deposit", "temp": 300},   # swapped but correct
        # clean is missing (omission)
    ]},
]

## 1. Ordered Matching (default)

Pairs by position. No `x-eval-align` needed.

In [36]:
ORDERED_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

result_ordered = evaluate(GOLD, EXTRACTED, schema=ORDERED_SCHEMA)

print(f"Mean F1: {result_ordered.mean_f1:.3f} Mean Precision: {result_ordered.mean_precision:.3f} Mean Recall: {result_ordered.mean_recall:.3f}")
print()
for record in result_ordered.records:
    show_record(record)
    print()

Mean F1: 0.333 Mean Precision: 0.333 Mean Recall: 0.333



Path,Gold,Extracted,Score,Status
steps[0].name,deposit,deposit,1.0,match
steps[0].temp,600,600,1.0,match


Path,Gold,Extracted,Score,Status
steps[0].name,deposit,anneal,0.0,mismatch
steps[0].temp,300,480,0.0,mismatch
steps[1].name,anneal,deposit,0.0,mismatch
steps[1].temp,500,300,0.0,mismatch
steps[-1].name,None,cool,0.0,hallucination
steps[-1].temp,None,50,0.0,hallucination


Path,Gold,Extracted,Score,Status
steps[0].name,deposit,anneal,0.0,mismatch
steps[0].temp,300,500,0.0,mismatch
steps[1].name,anneal,deposit,0.0,mismatch
steps[1].temp,500,300,0.0,mismatch
steps[2].name,clean,None,0.0,omission
steps[2].temp,100,None,0.0,omission


**Record 0** (ordered): Perfect match -- same element, same position.

**Record 1** (ordered): Pairs by position. Gold[0] `deposit` paired with extracted[0]
`anneal` -- name mismatch. Gold[1] `anneal` paired with extracted[1] `deposit` -- name
mismatch. The swap causes all fields to mismatch. `cool` is a hallucination.

**Record 2** (ordered): Same swap problem. Gold[0] `deposit` paired with extracted[0]
`anneal`, gold[1] `anneal` paired with extracted[1] `deposit` -- both name mismatches.
`clean` at gold[2] is an omission.

Ordered matching can't see that the right elements are present -- just in the wrong positions.

In [37]:
KEYFIELD_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "x-eval-align": {"match_by": "key_field", "key": "name"},
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

result_keyfield = evaluate(GOLD, EXTRACTED, schema=KEYFIELD_SCHEMA)

print(f"Mean F1: {result_keyfield.mean_f1:.3f}")
print()
for record in result_keyfield.records:
    show_record(record)
    print()

Mean F1: 0.800



Path,Gold,Extracted,Score,Status
steps[0].name,deposit,deposit,1.0,match
steps[0].temp,600,600,1.0,match


Path,Gold,Extracted,Score,Status
steps[0].name,deposit,deposit,1.0,match
steps[0].temp,300,300,1.0,match
steps[1].name,anneal,anneal,1.0,match
steps[1].temp,500,480,0.0,mismatch
steps[-1].name,None,cool,0.0,hallucination
steps[-1].temp,None,50,0.0,hallucination


Path,Gold,Extracted,Score,Status
steps[0].name,deposit,deposit,1.0,match
steps[0].temp,300,300,1.0,match
steps[1].name,anneal,anneal,1.0,match
steps[1].temp,500,500,1.0,match
steps[2].name,clean,None,0.0,omission
steps[2].temp,100,None,0.0,omission
